# EX_04 — Chatbots básicos (ejercicios)

**Notebook de referencia:** `notebook/04_Chatbots_Basicos.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Lista de mensajes

Construye `messages = [{"role": "system", ...}, {"role": "user", ...}]` para un asistente de estudio que **nunca** da la solución directa sino pistas.


In [1]:
messages = [
    {
        "role": "system",
        "content": (
            "Eres un tutor de estudio interactivo. Tu regla de oro es NUNCA dar la "
            "solución directa a los problemas o preguntas del estudiante. En su lugar, "
            "debes guiarlo utilizando el método socrático: ofrece pistas conceptuales, "
            "haz preguntas que lo hagan reflexionar y divide el problema en pasos más sencillos "
            "para que el estudiante descubra la respuesta por sí mismo. Mantén un tono paciente y motivador."
        )
    },
    {
        "role": "user",
        "content": "No entiendo este ejercicio de física, ¿me puedes dar la respuesta de cuánto da la fuerza?"
    }
]


## Actividad 2 — Historial acotado

Implementa una función `trim_history(messages, max_turns)` que conserve system + los últimos N intercambios user/assistant.


In [2]:
from typing import Any

def trim_history(messages: list[dict[str, Any]], max_turns: int) -> list[dict[str, Any]]:
    # 1. Si la lista está vacía, la devolvemos tal cual
    if not messages:
        return []
    
    # 2. Separamos el mensaje 'system' si existe en la primera posición
    has_system = messages[0].get("role") == "system"
    system_message = [messages[0]] if has_system else []
    
    # 3. El resto son los mensajes de la conversación (user/assistant)
    chat_messages = messages[1:] if has_system else messages
    
    # 4. Calculamos cuántos mensajes individuales mantener (1 turno = 2 mensajes)
    max_messages = max_turns * 2
    
    # 5. Tomamos los últimos N mensajes del historial de chat
    trimmed_chat = chat_messages[-max_messages:] if max_messages > 0 else []
    
    # 6. Reconstruimos la lista manteniendo el system prompt al inicio
    return system_message + trimmed_chat

# --- Ejemplo de uso para probarlo ---
historial_prueba = [
    {"role": "system", "content": "Eres un tutor."},
    {"role": "user", "content": "Hola 1"},
    {"role": "assistant", "content": "Hola 1"},
    {"role": "user", "content": "Hola 2"},
    {"role": "assistant", "content": "Hola 2"},
    {"role": "user", "content": "Hola 3"},
    {"role": "assistant", "content": "Hola 3"},
]

# Conservar solo los últimos 2 giros (intercambios 2 y 3)
resultado = trim_history(historial_prueba, max_turns=2)
print(f"Mensajes conservados: {len(resultado)}")
# Resultado esperado: System + 4 mensajes (User 2, Assistant 2, User 3, Assistant 3)

Mensajes conservados: 5


## Actividad 3 — Plantilla de resumen

Escribe un prompt que pida al modelo **resumir** el hilo cuando supere 800 palabras (pseudocódigo o string; puedes usar `len(text.split())`).


In [3]:
summarize_prompt = """
A continuación se muestra el historial de una conversación que ha superado el límite de longitud. 
Por favor, genera un resumen conciso y estructurado de la interacción que extraiga:
- Los temas clave discutidos.
- Las decisiones o conclusiones alcanzadas.
- Cualquier dato factual o técnico importante.

Elimina redundancias, saludos y charlas irrelevantes para mantener el resumen lo más compacto posible.

[HISTORIAL DE LA CONVERSACIÓN]
{conversation_history}

Resumen ejecutivo:
"""

# --- Lógica de activación en tu laboratorio ---

# 1. Unimos todo el contenido del historial actual para contar las palabras
full_text = " ".join([m["content"] for m in messages if m["role"] != "system"])

# 2. Si supera las 800 palabras, se dispara la compresión
if len(full_text.split()) > 800:
    # Formateamos el prompt inyectando el historial largo
    prompt_listo = summarize_prompt.format(conversation_history=full_text)
    
    # Aquí enviarías 'prompt_listo' al LLM para obtener el resumen.
    # El resultado ("Resumen ejecutivo...") reemplazaría los mensajes viejos 
    # en tu lista para liberar espacio de contexto (Memoria por Compresión).
    print("¡Alerta! El hilo supera las 800 palabras. Resumiendo...")

